# Resultats — Chargement et Evaluation des Modeles Generatifs

Charge les modeles entraines et analyse :
- Generation de nouvelles cartes SWI
- Espace latent (UMAP / t-SNE)
- Comparaison distributions reelles vs generees

> **Pre-requis** : `preprocessing.ipynb` et `train_cvae.ipynb` executes.

## 1. Imports et chargement des donnees

In [ ]:
import torch, torch.nn as nn, numpy as np, json
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from sklearn.manifold import TSNE
import warnings; warnings.filterwarnings('ignore')
try:
    import umap; HAS_UMAP = True
except ImportError:
    HAS_UMAP = False

MAPS_DIR   = Path('data/maps')
MODELS_DIR = Path('models')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')

In [ ]:
with open(MAPS_DIR / 'metadata.json') as f:
    meta = json.load(f)

H, W         = meta['H'], meta['W']
H_pad, W_pad = meta['H_pad'], meta['W_pad']
months_arr   = np.array(meta['months'], dtype=np.int32)
years_arr    = np.array(meta['years'],  dtype=np.int32)
monthly_stats = {int(k): tuple(v) for k, v in meta['monthly_stats'].items()}

swi_padded  = np.load(MAPS_DIR / 'swi_padded.npy')
swi_maps    = np.load(MAPS_DIR / 'swi_maps.npy')
land_mask   = np.load(MAPS_DIR / 'land_mask.npy')
mask_padded = np.load(MAPS_DIR / 'mask_padded.npy')
mask_tensor = torch.from_numpy(mask_padded)

T = len(months_arr)
print(f'Donnees : {swi_padded.shape}  |  T={T}')

## 2. Chargement du CVAE

In [ ]:
LATENT_DIM, COND_DIM = 32, 2

class Encoder(nn.Module):
    def __init__(self, H_pad, W_pad, ldim=LATENT_DIM, cdim=COND_DIM):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1,   32, 3, 2, 1), nn.GroupNorm(8,  32), nn.LeakyReLU(0.2),
            nn.Conv2d(32,  64, 3, 2, 1), nn.GroupNorm(16, 64), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 3, 2, 1), nn.GroupNorm(32,128), nn.LeakyReLU(0.2),
        )
        self.hb, self.wb = H_pad//8, W_pad//8
        flat = 128 * self.hb * self.wb
        self.fc   = nn.Sequential(nn.Linear(flat + cdim, 512), nn.LeakyReLU(0.2))
        self.mu   = nn.Linear(512, ldim)
        self.logv = nn.Linear(512, ldim)
    def forward(self, x, c):
        h = torch.cat([self.conv(x).flatten(1), c], 1)
        return self.mu(h := self.fc(h)), self.logv(h)

class Decoder(nn.Module):
    def __init__(self, H_pad, W_pad, ldim=LATENT_DIM, cdim=COND_DIM):
        super().__init__()
        self.hb, self.wb = H_pad//8, W_pad//8
        flat = 128 * self.hb * self.wb
        self.fc = nn.Sequential(
            nn.Linear(ldim + cdim, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, flat),        nn.LeakyReLU(0.2),
        )
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.GroupNorm(16,64), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(64,  32, 4, 2, 1), nn.GroupNorm(8, 32), nn.LeakyReLU(0.2),
            nn.ConvTranspose2d(32,   1, 4, 2, 1),
        )
    def forward(self, z, c):
        h = self.fc(torch.cat([z, c], 1))
        return self.deconv(h.view(h.size(0), 128, self.hb, self.wb))

class CVAE(nn.Module):
    def __init__(self, H_pad, W_pad):
        super().__init__()
        self.encoder = Encoder(H_pad, W_pad)
        self.decoder = Decoder(H_pad, W_pad)
    def reparameterize(self, mu, lv):
        return mu + torch.exp(0.5*lv)*torch.randn_like(mu) if self.training else mu
    def forward(self, x, c):
        mu, lv = self.encoder(x, c)
        return self.decoder(self.reparameterize(mu, lv), c), mu, lv

cvae = CVAE(H_pad, W_pad).to(device)
cvae.load_state_dict(torch.load(MODELS_DIR / 'cvae_swi.pth', map_location=device))
cvae.eval()
print('CVAE charge depuis models/cvae_swi.pth')

## 3. Generation de cartes

In [ ]:
@torch.no_grad()
def generate_maps(model, month, n=4):
    theta = 2 * np.pi * month / 12
    cond  = torch.tensor([[np.sin(theta), np.cos(theta)]]*n, dtype=torch.float32, device=device)
    z     = torch.randn(n, LATENT_DIM, device=device)
    out   = model.decoder(z, cond).cpu().numpy()
    mu_m, sig_m = monthly_stats[month]
    maps  = out[:, 0, :H, :W] * sig_m + mu_m
    maps[:, ~land_mask] = np.nan
    return maps

m_lbl = ['Jan','Fev','Mar','Avr','Mai','Jun','Jul','Aou','Sep','Oct','Nov','Dec']
mask_test = years_arr >= 2018

for month in [1, 8]:
    idx_r = np.random.choice(np.where(months_arr[mask_test] == month)[0], 4, replace=False)
    real  = swi_maps[np.where(mask_test)[0][idx_r]].copy(); real[:, ~land_mask] = np.nan
    gen   = generate_maps(cvae, month, 4)
    fig, axes = plt.subplots(2, 4, figsize=(16, 7))
    fig.suptitle(f'Reel vs Genere — {m_lbl[month-1]}', fontsize=13)
    for j in range(4):
        for row, data, t in [(0,real[j],f'Reel #{j+1}'),(1,gen[j],f'Genere #{j+1}')]:
            im = axes[row,j].imshow(data, cmap='RdYlBu', origin='lower', vmin=0, vmax=1.3, aspect='auto')
            axes[row,j].set_title(t, fontsize=9); axes[row,j].axis('off')
    plt.colorbar(im, ax=axes, label='SWI', fraction=0.015, pad=0.04)
    plt.tight_layout(); plt.show()

## 4. Espace latent

In [ ]:
@torch.no_grad()
def encode_all(model, maps, months, bs=32):
    theta = 2 * np.pi * months / 12
    cond_all = np.stack([np.sin(theta), np.cos(theta)], 1).astype(np.float32)
    mus = []
    for i in range(0, len(maps), bs):
        xb = torch.from_numpy(maps[i:i+bs, None]).to(device)
        cb = torch.from_numpy(cond_all[i:i+bs]).to(device)
        mu, _ = model.encoder(xb, cb); mus.append(mu.cpu().numpy())
    return np.vstack(mus)

latent_z = encode_all(cvae, swi_padded, months_arr)
reducer  = umap.UMAP(n_components=2, random_state=42) if HAS_UMAP else TSNE(n_components=2, random_state=42)
z_2d     = reducer.fit_transform(latent_z)
method   = 'UMAP' if HAS_UMAP else 't-SNE'
print(f'{method} : {z_2d.shape}')

In [ ]:
swi_mean = np.nanmean(swi_maps, axis=(1,2))
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'Espace latent CVAE ({method})', fontsize=14)
kw = dict(s=10, alpha=0.7, linewidths=0)
sc = axes[0].scatter(z_2d[:,0], z_2d[:,1], c=months_arr, cmap='hsv', **kw)
plt.colorbar(sc, ax=axes[0], label='Mois', ticks=range(1,13)); axes[0].set_title('(A) Mois')
sc = axes[1].scatter(z_2d[:,0], z_2d[:,1], c=years_arr,  cmap='plasma', **kw)
plt.colorbar(sc, ax=axes[1], label='Annee'); axes[1].set_title('(B) Annee')
p5, p95 = np.percentile(swi_mean, [5,95])
sc = axes[2].scatter(z_2d[:,0], z_2d[:,1], c=swi_mean, cmap='RdYlBu', vmin=p5, vmax=p95, **kw)
plt.colorbar(sc, ax=axes[2], label='SWI moyen'); axes[2].set_title('(C) SWI moyen')
for ax in axes: ax.set_xlabel(method+'-1'); ax.set_ylabel(method+'-2'); ax.grid(True, alpha=0.15)
plt.tight_layout(); plt.show()

## 5. Distributions marginales

In [ ]:
m_lbl = ['Jan','Fev','Mar','Avr','Mai','Jun','Jul','Aou','Sep','Oct','Nov','Dec']
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.suptitle('Distribution SWI : Reel vs Genere', fontsize=14)
for m in range(1, 13):
    ax = axes[(m-1)//4, (m-1)%4]
    rv = swi_maps[months_arr==m][:, land_mask].flatten()
    rv = rv[np.isfinite(rv)]
    gv = generate_maps(cvae, m, n=20)[:, land_mask].flatten()
    gv = gv[np.isfinite(gv)]
    ax.hist(rv, bins=40, alpha=0.55, density=True, color='steelblue', label='Reel')
    ax.hist(gv, bins=40, alpha=0.55, density=True, color='tomato',    label='Genere')
    ax.set_title(m_lbl[m-1]); ax.set_xlim([-0.05, 1.55]); ax.grid(True, alpha=0.2)
    if m==1: ax.legend(fontsize=8)
plt.tight_layout(); plt.show()